# Step 3 — Build the Historical Weather Panel

**Objective:** Create a county x day weather panel from NOAA ISD (2018-2024) and derive compound event flags (heatwave, compound heat-wind, compound triple).

**Output:** `data/processed/weather_panel_ercot.csv` and `weather_panel_caiso.csv`

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from config.settings import RAW, PROCESSED, ERCOT_FIPS, CAISO_FIPS, HIST_START_YEAR, HIST_END_YEAR
from src.data.noaa_isd import build_weather_panel
from src.analysis.compound_flags import add_weather_flags


## 3.1 Load county geometries (from TIGER/Line, built in Step 1)

In [ ]:
import geopandas as gpd

tiger_path = RAW['hifld'] / 'tl_2023_us_county' / 'tl_2023_us_county.shp'
if tiger_path.exists():
    counties = gpd.read_file(tiger_path)
    counties['fips'] = counties['GEOID'].astype(str).str.zfill(5)
    ercot_gdf = counties[counties['fips'].isin(set(ERCOT_FIPS))].copy()
    caiso_gdf = counties[counties['fips'].isin(set(CAISO_FIPS))].copy()
    print('Shapefiles loaded.')
else:
    print('Shapefile not found.')
    ercot_gdf = caiso_gdf = None


## 3.2 Build ERCOT weather panel

ISD station files are downloaded from `s3://noaa-isd-pds/` and cached locally. Requires internet + `boto3`.

In [ ]:
if ercot_gdf is not None:
    try:
        ercot_weather = build_weather_panel(
            region_fips=ERCOT_FIPS,
            county_gdf=ercot_gdf,
            cache_dir=RAW['noaa_isd'],
            year_range=(HIST_START_YEAR, HIST_END_YEAR),
        )
        print(f'ERCOT weather panel: {ercot_weather.shape}')
        print(ercot_weather.head())
    except Exception as e:
        print(f'Weather build failed: {e}')
        ercot_weather = None
else:
    ercot_weather = None


## 3.3 Derive compound event flags

In [ ]:
if ercot_weather is not None:
    ercot_weather = add_weather_flags(ercot_weather)
    flag_cols = ['heatwave_day', 'heatwave_event', 'compound_heat_wind',
                 'compound_heat_precip', 'compound_triple']
    present = [c for c in flag_cols if c in ercot_weather.columns]
    print('Flag rates (fraction of county-days):')
    print(ercot_weather[present].mean())


## 3.4 NOAA Storm Events supplement

Download bulk CSV from: https://www.ncei.noaa.gov/stormevents/ftp.jsp
Save to `data/raw/storm_events/`.

In [ ]:
storm_files = list(RAW['storm_events'].glob('StormEvents_details*.csv'))
print(f'Found {len(storm_files)} Storm Events files')
if storm_files:
    se = pd.concat([pd.read_csv(f, low_memory=False) for f in storm_files], ignore_index=True)
    print(f'Total rows: {len(se):,}')
    print(se['EVENT_TYPE'].value_counts().head(10))


## 3.5 Save processed weather panels

In [ ]:
if ercot_weather is not None:
    ercot_weather.to_csv(PROCESSED['weather_ercot'])
    print('Saved:', PROCESSED['weather_ercot'])
